In [5]:

import datetime
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

data_set = pd.read_csv('Churn_Modelling.csv')

data_set = data_set.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

#gender
label_encoder_gender = LabelEncoder()
data_set['Gender'] = label_encoder_gender.fit_transform(data_set['Gender'])

#geography
onehotencoded_geography = OneHotEncoder()
geo_encoded = onehotencoded_geography.fit_transform(data_set[['Geography']]).toarray()
geo_names = onehotencoded_geography.get_feature_names_out(['Geography'])
geo_df = pd.DataFrame(geo_encoded, columns=geo_names)

data_set = pd.concat([data_set.drop('Geography', axis=1), geo_df], axis=1)

X = data_set.drop('Exited', axis=1)
Y = data_set['Exited']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with open('gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('geography.pkl', 'wb') as file:
    pickle.dump(onehotencoded_geography, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

seq_model = tf.keras.Sequential([
    Dense(64, activation='relu', input_shape =(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

_opt = tf.keras.optimizers.Adam(learning_rate=0.001)
_loss = tf.keras.losses.BinaryCrossentropy()
_metrics = tf.keras.metrics.BinaryAccuracy()

seq_model.compile(optimizer = _opt, loss = _loss, metrics = [_metrics])

logs_dir = "logs/fit/" +datetime.datetime.now().strftime('%d-%m-%Y_%H-%M-%S')
tensorflow_callback = tf.keras.callbacks.TensorBoard(log_dir = logs_dir, histogram_freq =1)
earlystopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


seq_model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=50, callbacks=[tensorflow_callback, earlystopping_callback])
seq_model.save('churn_model.keras')

%load_ext tensorboard
%tensorboard --logdir logs/fit

Epoch 1/50


c:\Users\bdn7kor\Documents\Projects\Courses\ANN\env2\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.6764 - loss: 165.4715 - val_binary_accuracy: 0.6935 - val_loss: 74.8132
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6770 - loss: 115.6805 - val_binary_accuracy: 0.7975 - val_loss: 100.3077
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6836 - loss: 107.1705 - val_binary_accuracy: 0.4790 - val_loss: 69.3978
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6804 - loss: 80.5730 - val_binary_accuracy: 0.4800 - val_loss: 61.0407
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6824 - loss: 90.1606 - val_binary_accuracy: 0.5125 - val_loss: 38.7220
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6764 - loss: 84.6240 - val_binary_accuracy: 0.5850 - val_loss: 31.9278
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6771 - loss: 65.5166 - val_binary_accuracy: 0.5545 - val_loss: 19.1735

In [6]:
%reload_ext tensorboard
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 26172), started 0:07:02 ago. (Use '!kill 26172' to kill it.)

In [44]:
input_data = {
    "CreditScore": 600,
    "Geography": "Germany",
    "Gender": "Female",
    "Age": 29,
    "Tenure": 3,
    "Balance": 60,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000
}

with open('gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('geography.pkl', 'rb') as file:
    onehotencoded_geography = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

input_data_df = pd.DataFrame([input_data])

#gender
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])

#geography
geo_encoded = onehotencoded_geography.transform(input_data_df[['Geography']]).toarray()
geo_names = onehotencoded_geography.get_feature_names_out(['Geography'])
geo_df = pd.DataFrame(geo_encoded, columns=geo_names)

input_data_df = pd.concat([input_data_df.drop('Geography', axis=1), geo_df], axis=1)

#scaling
input_data_df_scaled = scaler.transform(input_data_df)

loaded_model =tf.keras.models.load_model('churn_model.keras')
prediction = loaded_model.predict(input_data_df_scaled)
prediction_probability = prediction[0][0]


if prediction_probability > 0.5:
    print(f"The customer is likely to churn with a probability of {prediction_probability:.2f}.")  
else:
    print(f"The customer is unlikely to churn with a probability of {prediction_probability:.2f}.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
The customer is unlikely to churn with a probability of 0.02.
